In [1]:
import pandas as pd
import re
from collections import defaultdict


file_path = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026 2.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data ")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master ")


for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# =============================
# CYCLE TIME (SECONDS → MINUTES)
# =============================
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =============================
# CONSTANTS
# =============================
ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
MACHINE_CAPACITY = 22 * 60  # 1320 minutes (1 day)

# =============================
# MACHINE NORMALIZER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# STEP 1: FILTER DAILY PLAN > 0
# =============================
valid = master[master["Daily Plan"] > 0].copy()

# =============================
# STEP 2: AGGREGATE AT CHILD PART LEVEL
# =============================
records = []

for child, grp in valid.groupby("Child Part", sort=False):

    demand_from_switches = (grp["Daily Plan"] * grp["Sub Count"]).sum()
    min_qty = grp["Minimum Quantity"].iloc[0]
    inventory = grp["Inventory_25"].iloc[0]

    machines = ",".join(grp["Vertical Machines"].astype(str))

    records.append({
        "Child Part": child,
        "Required Qty": min_qty + demand_from_switches,
        "Inventory_25": inventory,
        "Vertical Machines": machines
    })

agg = pd.DataFrame(records)
agg["Net Required Qty"] = agg["Required Qty"] - agg["Inventory_25"]

# =============================
# TRACKING
# =============================
machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)

# =============================
# STEP 3: ASSIGN ALL QUANTITY (NO CAPACITY CHECK)
# =============================
for _, row in agg.iterrows():

    child = row["Child Part"]
    net_qty = row["Net Required Qty"]

    if net_qty <= 0:
        continue

    ct = cycle_time_min.get(child)
    if not ct or ct <= 0:
        continue

    total_time = net_qty * ct

    # Parse machines in given order
    raw = str(row["Vertical Machines"])
    tokens = re.split(r"[,\|/\\\n]+", raw)

    eligible = []
    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES and m not in eligible:
            eligible.append(m)

    if not eligible:
        continue

    # 🔥 Assign EVERYTHING to first eligible machine
    m = eligible[0]

    machine_load[m] += total_time

    machine_plan[m].append({
        "Child Part": child,
        "Quantity": round(net_qty, 2),
        "Time Used (min)": round(total_time, 2)
    })

# =============================
# DISPLAY RESULTS
# =============================
print("\n========== MACHINE-WISE PLAN (1 DAY, NO CAPACITY LIMIT) ==========\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    if machine_plan[m]:
        display(pd.DataFrame(machine_plan[m]))
    else:
        print("No allocation")
    print("-" * 60)

print("\n========== MACHINE LOAD SUMMARY (OVERLOAD VISIBLE) ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Capacity (min)": MACHINE_CAPACITY,
        "Overload (min)": round(machine_load[m] - MACHINE_CAPACITY, 2)
    }
    for m in ALLOWED_MACHINES
]))


========== MACHINE-WISE PLAN (1 DAY, NO CAPACITY LIMIT) ==========

🔧 MP-01


,Child Part,Quantity,Time Used (min)
0,S11434-002A0X,816.52,612.39
1,S12071-001A0X,41.35,24.12
2,S12094-003A0X,22.68,18.14
3,S41354-011A0X,2434.84,1826.13


------------------------------------------------------------
🔧 MP-05


,Child Part,Quantity,Time Used (min)
0,14SW030082-00001X0,689.71,517.28
1,14SW220197-00005X0,3399.84,1416.60
2,S12095-004A0X,1713.26,1284.94
3,S13083-003A0X,3283.61,1641.81
4,S13108-002A0X,18095.10,7690.42


------------------------------------------------------------
🔧 MP-10


,Child Part,Quantity,Time Used (min)
0,S11398-018A0X,187.97,140.98
1,S11398-030A0X,589.00,589.00
2,S13080-004A0X,97.52,37.38
3,S22166-016A0X,394.03,210.15
4,S33107-005A0X,108.29,54.15


------------------------------------------------------------
🔧 MP-17


,Child Part,Quantity,Time Used (min)
0,14SW110487-00009X0,285.26,114.10
1,14SW220201-00005X0,1152.10,691.26


------------------------------------------------------------

========== MACHINE LOAD SUMMARY (OVERLOAD VISIBLE) ==========



,Machine,Used (min),Capacity (min),Overload (min)
0,MP-01,2480.78,1320,1160.78
1,MP-05,12551.05,1320,11231.05
2,MP-10,1031.65,1320,-288.35
3,MP-17,805.36,1320,-514.64


In [1]:
import pandas as pd
import re
from collections import defaultdict

file_path = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026 2.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data ")
ppm   = pd.read_excel(file_path, sheet_name="Part Production Master ")

for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)
ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

cycle_time_min = {row["Material"]: row["Machine"] / 60 for _, row in ppm.iterrows()}

ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
MACHINE_CAPACITY = 22 * 60
CHANGEOVER_TIME  = 40
MAX_FILL_UTIL    = 0.98   # stop at 98%

def normalize_machine(m):
    if not m or pd.isna(m): return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# Prepare data
valid = master[master["Daily Plan"] > 0].copy()
records = []
for child, grp in valid.groupby("Child Part", sort=False):
    daily = (grp["Daily Plan"] * grp["Sub Count"]).sum()
    min_qty = grp["Minimum Quantity"].iloc[0]
    inv = grp["Inventory_25"].iloc[0]
    machines = ",".join(grp["Vertical Machines"].astype(str))
    records.append({
        "Child Part": child,
        "Net Required": daily + min_qty - inv,
        "Vertical Machines": machines
    })

df = pd.DataFrame(records)
df["Net Required"] = df["Net Required"].clip(lower=0)

# Tracking
machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)
machine_parts = defaultdict(set)
assigned_qty = defaultdict(float)

# ============================= PHASE 1: Balanced Assignment
for _, row in df.iterrows():
    child = row["Child Part"]
    qty = row["Net Required"]
    if qty <= 0: continue
    ct = cycle_time_min.get(child, 0)
    if ct <= 0: continue

    remaining = qty
    tokens = re.split(r"[,\|/\\\n]+", str(row["Vertical Machines"]))
    eligible = [normalize_machine(t) for t in tokens if normalize_machine(t) in ALLOWED_MACHINES]
    eligible = list(dict.fromkeys(eligible))

    if not eligible: continue

    eligible = sorted(eligible, key=lambda m: MACHINE_CAPACITY - machine_load[m], reverse=True)

    for m in eligible:
        if remaining <= 0: break
        avail = MACHINE_CAPACITY - machine_load[m]
        co = CHANGEOVER_TIME if child not in machine_parts[m] else 0
        if avail <= co: continue
        avail -= co
        max_fit = avail / ct
        if max_fit <= 0: continue

        assign = min(remaining, max_fit)
        time_prod = assign * ct

        machine_load[m] += co + time_prod
        if co > 0: machine_parts[m].add(child)
        remaining -= assign
        assigned_qty[child] += assign

        machine_plan[m].append({
            "Child Part": child,
            "Quantity": round(assign, 2),
            "Prod Time": round(time_prod, 2),
            "Changeover": round(co, 2) if co > 0 else 0,
            "Phase": "1-Balanced"
        })

# ============================= Build remaining list (120T only)
remaining_120t = []
assigned_to_120t = set()
for m in ALLOWED_MACHINES:
    for e in machine_plan[m]:
        assigned_to_120t.add(e["Child Part"])

for child in assigned_to_120t:
    req = df[df["Child Part"] == child]["Net Required"].iloc[0]
    ass = assigned_qty.get(child, 0)
    rem = req - ass
    if rem > 0.01:
        remaining_120t.append({
            "Child Part": child,
            "Net Required": round(req, 2),
            "Assigned": round(ass, 2),
            "Remaining": round(rem, 2)
        })

remaining_120t.sort(key=lambda x: x["Remaining"], reverse=True)

# ============================= PHASE 2: Fill low-util machines to 95–98%
for entry in remaining_120t:
    child = entry["Child Part"]
    remaining = entry["Remaining"]
    if remaining <= 0: continue

    ct = cycle_time_min.get(child, 0)
    if ct <= 0: continue

    row = df[df["Child Part"] == child].iloc[0]
    tokens = re.split(r"[,\|/\\\n]+", str(row["Vertical Machines"]))
    eligible = [normalize_machine(t) for t in tokens if normalize_machine(t) in ALLOWED_MACHINES]
    eligible = list(dict.fromkeys(eligible))

    while remaining > 0.01:
        # Candidates: machines that can still take more (under 98%)
        candidates = [(m, machine_load[m]) for m in eligible
                      if machine_load[m] / MACHINE_CAPACITY < MAX_FILL_UTIL]

        if not candidates: break

        # Pick least loaded
        candidates.sort(key=lambda x: x[1])
        m = candidates[0][0]

        avail = MACHINE_CAPACITY * MAX_FILL_UTIL - machine_load[m]
        co = CHANGEOVER_TIME if child not in machine_parts[m] else 0
        if avail <= co: continue
        avail -= co

        max_fit = avail / ct
        assign = min(remaining, max_fit)
        if assign <= 0: continue

        time_prod = assign * ct

        machine_load[m] += co + time_prod
        if co > 0: machine_parts[m].add(child)
        remaining -= assign
        assigned_qty[child] += assign

        machine_plan[m].append({
            "Child Part": child,
            "Quantity": round(assign, 2),
            "Prod Time": round(time_prod, 2),
            "Changeover": round(co, 2) if co > 0 else 0,
            "Phase": "2-Fillup"
        })

    entry["Remaining"] = round(remaining, 2)
    entry["Assigned"] = round(entry["Net Required"] - remaining, 2)

# ============================= Final Output
print("\n===== FINAL MACHINE PLAN (Balanced + Fill-up to 95-98%) =====\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    if machine_plan[m]:
        print(pd.DataFrame(machine_plan[m]))
    else:
        print("No allocation")
    util = round(100 * machine_load[m] / MACHINE_CAPACITY, 1)
    print(f"→ Utilization: {util}% ({round(machine_load[m],1)} / 1320 min)\n{'-'*70}")

print("\n===== MACHINE SUMMARY =====\n")
print(pd.DataFrame([
    {"Machine": m, "Used": round(machine_load[m],1), "Util %": round(100*machine_load[m]/MACHINE_CAPACITY,1)}
    for m in ALLOWED_MACHINES
]))

print("\n===== REMAINING AFTER FILL-UP =====\n")
if remaining_120t:
    print(pd.DataFrame(remaining_120t))
    total_left = sum(x["Remaining"] for x in remaining_120t)
    print(f"\nTotal qty still left to plan: {round(total_left, 2)} pcs")
else:
    print("Nothing left — all possible qty assigned to 95–98% utilization!")


===== FINAL MACHINE PLAN (Balanced + Fill-up to 95-98%) =====

🔧 MP-01
      Child Part  Quantity  Prod Time  Changeover       Phase
0  S11434-002A0X    816.52     612.39          40  1-Balanced
1  S12071-001A0X     41.35      24.12          40  1-Balanced
2  S12094-003A0X     22.68      18.14          40  1-Balanced
3  S41354-011A0X    673.80     505.35          40  1-Balanced
→ Utilization: 100.0% (1320.0 / 1320 min)
----------------------------------------------------------------------
🔧 MP-05
           Child Part  Quantity  Prod Time  Changeover       Phase
0  14SW030082-00001X0    689.71     517.28          40  1-Balanced
1  14SW220197-00005X0   1734.52     722.72          40  1-Balanced
→ Utilization: 100.0% (1320.0 / 1320 min)
----------------------------------------------------------------------
🔧 MP-10
      Child Part  Quantity  Prod Time  Changeover       Phase
0  S11398-018A0X    187.97     140.98          40  1-Balanced
1  S11398-030A0X    589.00     589.00          40  